In [ ]:
import os
import matplotlib.pyplot as plt
import pytorch_lightning as pl
import torch
from retrieval.knn_seg_hbird import KNNHummingBirdSegmentation
from argparse import Namespace
from pretrain.mae.lit_mae import LitMAE
import segmentation.utils as utils
from data.retrieval_module import RetrievalDataModule
from data.semantic_module import SemanticDataModule
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold
import albumentations as A
import seaborn as sns
import cv2

%matplotlib inline
%load_ext autoreload

REPO_ROOT = "../"
os.chdir(os.path.dirname(REPO_ROOT))

# Set random seeds
seed = 42
pl.seed_everything(seed, workers=True)

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Enhanced thesis-ready configuration
plt.rcParams['font.size'] = 14          # Slightly larger base font
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 18     # More prominent titles
plt.rcParams['axes.labelsize'] = 14     # Clearer axis labels
plt.rcParams['xtick.labelsize'] = 12    # Readable tick labels
plt.rcParams['ytick.labelsize'] = 12    # Readable tick labels
plt.rcParams['legend.fontsize'] = 12    # Add legend font size
plt.rcParams['figure.titlesize'] = 20   # Overall figure title
plt.rcParams['lines.linewidth'] = 2     # Thicker lines for clarity
plt.rcParams['axes.linewidth'] = 1.2    # Thicker axes
plt.rcParams['grid.alpha'] = 0.3        # Subtle grid if used
plt.rcParams['savefig.dpi'] = 300       # High resolution for print
plt.rcParams['savefig.bbox'] = 'tight'  # Remove extra whitespace
plt.rcParams['figure.figsize'] = (8, 6) # Good default size

# MAE Results

### Initialization

In [ ]:
os.sync()
with open(f"{REPO_ROOT}/retrieval/knn_seg_hbird.py", "r") as f:
    t = f.read()
    print(t)

with open(f"{REPO_ROOT}/data/retrieval_dataset.py", "r") as f:
    t = f.read()
    print(t)


%autoreload 2
config_path = "./configs/finetune/finetune_main.yml"
model_config_path = "./configs/finetune/finetune_knn_test.yml"

args = Namespace()

config = utils.merge_config(config_path, args, model_config_path)
config.model_params = utils.merge_model_params_segementation_encoder(
    config, config.encoder_ckpt_model_params)
config.data_path = "./datasets/pretrain_split/"
config.encoder_ckpt_path = f"{REPO_ROOT}/scripts/logs/31108_good_epoch=6158-val_loss=0.003.ckpt"
config.input_resolution = [512, 512]

print("Configuration after merge: ", config)

mae = LitMAE.load_from_checkpoint(
    config.encoder_ckpt_path, parameters=config.model_params,
    strict=False,
)

config.normalize = False

encoder = mae.get_encoder()
torch.cuda.memory_allocated() / (1024**2)

## Average

In [ ]:
retrieval_module = RetrievalDataModule(
    data_path="./datasets/pretrain_split/",
    batch_size=32,
    classes=config.classes,
    num_workers=4,
    input_resolution=config.input_resolution,
    train_size=0.8,
)

retrieval_module.setup("train")

train_loader = retrieval_module.train_dataloader()
val_loader = retrieval_module.val_dataloader()

In [ ]:
knn_evaluator = KNNHummingBirdSegmentation(
    encoder=encoder,
    normalize=config.normalize,
    train_loader=train_loader,
    val_loader=val_loader,
    classes=config.classes,
)

In [ ]:
class_map ={
    0: "wire",
    1: "ball",
    2: "wedge",
    3: "epoxy"
}

def evaluate_with_kfold(encoder, config, k_folds=5, random_state=42, 
                        num_classes=4):
    """
    Use K-fold cross validation for each hyperparameter combination
    """
    
    # Get all images
    data_path = Path(config.data_path)
    im_list = [data_path / "train" / img_file for img_file in os.listdir(data_path / "train")]
    
    # Create K-fold splits
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)
    
    # Hyperparameter grid
    thresholds = [0.5, 0.2, 0.1, 0.05, 0.01]
    betas = [0.3, 0.15, 0.1, 0.02, 0.002]
    ks = [2, 10, 20, 30, 50]
    
    all_results = []
    
    for k in ks:
        for threshold in thresholds:
            for beta in betas:
                print(f"\n=== Testing k={k}, threshold={threshold}, beta={beta} ===")
                
                fold_results = {cls: [] for cls in config.classes}
                
                # K-fold evaluation
                for fold, (train_indices, val_indices) in enumerate(kf.split(im_list)):
                    print(f"  Fold {fold + 1}/{k_folds}")
                    
                    # Create fold-specific image lists
                    train_im_list = [im_list[i] for i in train_indices]
                    val_im_list = [im_list[i] for i in val_indices]
                    
                    # Create evaluator for this fold
                    fold_evaluator = KNNHummingBirdSegmentation(
                        encoder=encoder,
                        train_im_list=train_im_list,
                        val_im_list=val_im_list,
                        num_batches=-1,
                        profile_time=True,
                        normalize=False,
                        average_patches=False
                    )
                    
                    # Evaluate
                    m_ious = fold_evaluator.evaluate(
                        thresholds=[threshold]*num_classes,
                        betas=beta,
                        k=k,
                        patch_mechanism="complete",
                    )
                    
                    
                    # Store results
                    for cls in class_map.values():
                        if m_ious[cls] is not None:
                            fold_results[cls].append(m_ious[cls])
                    
                    del fold_evaluator
                    torch.cuda.empty_cache()
                
                # Calculate statistics
                for cls in class_map.values():
                    class_scores = fold_results[cls]
                    all_results.append({
                        'class': cls,
                        'threshold': threshold,
                        'beta': beta,
                        'fold_scores': class_scores,
                        'mean_miou': np.mean(class_scores) if class_scores else None,
                        'std_miou': np.std(class_scores) if class_scores else None,
                        'cv_score': np.mean(class_scores) - np.std(class_scores) if class_scores else None  # Conservative estimate
                    })
        
        return pd.DataFrame(all_results)

# Run K-fold evaluation
results_df = evaluate_with_kfold(encoder, config, k_folds=5, random_state=42)

In [ ]:
results_df.to_csv("mae_hbird_kfold_evaluation_results.csv", index=False)

In [ ]:
results_df = pd.read_csv("mae_hbird_kfold_evaluation_results.csv")

results_df.groupby('class').max('mean_miou').sort_values(by="mean_miou", ascending=False)

In [ ]:
grouped_results =  results_df.groupby(['class', 'threshold', 'beta']).\
    max('mean_miou')["mean_miou"].\
    reset_index()

class_map ={
    0: "Wire",
    1: "Ball",
    2: "Wedge",
    3: "Epoxy"
}

for cls, name in [(c, class_map[c]) for c in grouped_results['class'].unique()]:
    data = grouped_results[grouped_results['class'] == cls]
    heatmap_data = data.pivot(index='threshold', columns='beta', values='mean_miou')
    plt.figure(figsize=(8, 6))
    sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="viridis")
    plt.title(f"Class {name} - Mean mIoU Heatmap")
    plt.xlabel("Beta")
    plt.ylabel("Class Threshold")
    plt.savefig(f"visualizations/hbird_hyper_heatmap_class_{cls}.pdf", bbox_inches='tight')
    plt.show()

## Complete

In [ ]:
os.sync()
with open(f"{REPO_ROOT}/retrieval/knn_seg_hbird.py", "r") as f:
    t = f.read()
    print(t)

with open(f"{REPO_ROOT}/data/retrieval_dataset.py", "r") as f:
    t = f.read()
    print(t)


%autoreload 2
config_path = "./configs/finetune/finetune_main.yml"
model_config_path = "./configs/finetune/finetune_knn_test.yml"

args = Namespace()

config = utils.merge_config(config_path, args, model_config_path)

config.model_params = utils.merge_model_params_segementation_encoder(
    config, config.encoder_ckpt_model_params)

#config.data_path = "./datasets/finetune/epoxy/0.6.57/"
config.data_path = "./datasets/pretrain_split/"

config.encoder_ckpt_path = f"{REPO_ROOT}/scripts/logs/31108_good_epoch=6158-val_loss=0.003.ckpt"
#config.ckpt_path = f"{REPO_ROOT}/checkpoints/pretrain-vit-dino-more-augmentations/aoi-pretrain-dino_40012/epoch=239-val_knn_mean_iou=0.506.ckpt"

print("Configuration after merge: ", config)

mae = LitMAE.load_from_checkpoint(
    config.encoder_ckpt_path, parameters=config.model_params,
    strict=False
)

config.normalize = False

encoder = mae.get_encoder()
torch.cuda.memory_allocated() / (1024**2)

In [ ]:
class_map ={
    0: "wire",
    1: "ball",
    2: "wedge",
    3: "epoxy"
}

def evaluate_with_kfold(encoder, config, k_folds=5, random_state=42, 
                        num_classes=4):
    """
    Use K-fold cross validation for each hyperparameter combination
    """
    
    # Get all images
    data_path = Path(config.data_path)
    im_list = [data_path / "train" / img_file for img_file in os.listdir(data_path / "train")]
    
    # Create K-fold splits
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)
    
    # Hyperparameter grid
    thresholds = [0.5, 0.2, 0.1, 0.05, 0.01]
    betas = [0.3, 0.15, 0.1, 0.02, 0.002]
    ks = [2, 10, 20, 30, 50]
    
    all_results = []
    
    for k in ks:
        for threshold in thresholds:
            for beta in betas:
                print(f"\n=== Testing k={k}, threshold={threshold}, beta={beta} ===")
                
                fold_results = {cls: [] for cls in config.classes}
                
                # K-fold evaluation
                for fold, (train_indices, val_indices) in enumerate(kf.split(im_list)):
                    print(f"  Fold {fold + 1}/{k_folds}")
                    
                    # Create fold-specific image lists
                    train_im_list = [im_list[i] for i in train_indices]
                    val_im_list = [im_list[i] for i in val_indices]
                    
                    # Create evaluator for this fold
                    fold_evaluator = KNNHummingBirdSegmentation(
                        encoder=encoder,
                        train_im_list=train_im_list,
                        val_im_list=val_im_list,
                        num_batches=-1,
                        profile_time=True,
                        normalize=False,
                        average_patches=False
                    )
                    
                    # Evaluate
                    m_ious = fold_evaluator.evaluate(
                        thresholds=[threshold]*num_classes,
                        betas=beta,
                        k=k,
                        patch_mechanism="complete",
                    )
                    
                    
                    # Store results
                    for cls in class_map.values():
                        if m_ious[cls] is not None:
                            fold_results[cls].append(m_ious[cls])
                    
                    del fold_evaluator
                    torch.cuda.empty_cache()
                
                # Calculate statistics
                for cls in class_map.values():
                    class_scores = fold_results[cls]
                    all_results.append({
                        'class': cls,
                        'threshold': threshold,
                        'beta': beta,
                        'fold_scores': class_scores,
                        'mean_miou': np.mean(class_scores) if class_scores else None,
                        'std_miou': np.std(class_scores) if class_scores else None,
                        'cv_score': np.mean(class_scores) - np.std(class_scores) if class_scores else None  # Conservative estimate
                    })
        
        return pd.DataFrame(all_results)

# Run K-fold evaluation
results_df = evaluate_with_kfold(encoder, config, k_folds=5, random_state=42)

In [ ]:
results_df.to_csv("mae_hbird_complete_patches_kfold_evaluation_results.csv", index=False)

In [ ]:
results_df = pd.read_csv("mae_hbird_complete_patches_kfold_evaluation_results.csv")

results_df.groupby('class').max('mean_miou').sort_values(by="mean_miou", ascending=False)

## Distance

In [ ]:
import pandas as pd

class_map ={
    0: "iou_wire",
    1: "iou_ball",
    2: "iou_wedge",
    3: "iou_epoxy"
}

def evaluate_with_kfold(encoder, config, k_folds=5, random_state=42, 
                        num_classes=4):
    """
    Use K-fold cross validation for each hyperparameter combination
    """
    
    # Get all images
    data_path = Path(config.data_path)
    im_list = [data_path / "train" / img_file for img_file in os.listdir(data_path / "train")]
    
    # Create K-fold splits
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)
    
    # Hyperparameter grid
    thresholds = [0.5, 0.2, 0.1, 0.05, 0.01]
    ks = [2, 4, 6, 10, 20, 30]
    
    all_results = []
    
    for k in ks:
        for threshold in thresholds:
                print(f"\n=== Testing k={k}, threshold={threshold} ===")
                
                fold_results = {f"iou_{cls}": [] for cls in config.classes}
                
                # K-fold evaluation
                for fold, (train_indices, val_indices) in enumerate(kf.split(im_list)):
                    print(f"  Fold {fold + 1}/{k_folds}")
                    
                    # Create fold-specific image lists
                    train_im_list = [im_list[i] for i in train_indices]
                    val_im_list = [im_list[i] for i in val_indices]
                    
                    # Create evaluator for this fold
                    fold_evaluator = KNNHummingBirdSegmentation(
                        encoder=encoder,
                        train_im_list=train_im_list,
                        val_im_list=val_im_list,
                        num_batches=-1,
                        profile_time=True,
                        normalize=False
                    )
                    
                    # Evaluate
                    m_ious = fold_evaluator.evaluate_distance(
                        threshold=threshold,
                        k=k,
                    )
                    
                    print(m_ious)
                    print(class_map.values())
                    # Store results
                    for cls in class_map.values():
                        if m_ious[cls] is not None:
                            fold_results[cls].append(m_ious[cls])
                    
                    del fold_evaluator
                    torch.cuda.empty_cache()
                
                # Calculate statistics
                for cls in class_map.values():
                    class_scores = fold_results[cls]
                    all_results.append({
                        'class': cls,
                        'k': k,
                        'threshold': threshold,
                        'fold_scores': class_scores,
                        'mean_miou': np.mean(class_scores) if class_scores else None,
                        'std_miou': np.std(class_scores) if class_scores else None,
                        'cv_score': np.mean(class_scores) - np.std(class_scores) if class_scores else None  # Conservative estimate
                    })
        
        return pd.DataFrame(all_results)

# Run K-fold evaluation
results_df = evaluate_with_kfold(encoder, config, k_folds=5, random_state=42)

In [ ]:
results_df.to_csv("mae_hbird_distance_kfold_evaluation_results.csv", index=False)

In [ ]:
results_df = pd.read_csv("mae_hbird_distance_kfold_evaluation_results.csv")

results_df.groupby('class').max('mean_miou').sort_values(by="mean_miou", ascending=False)

# Semantic Dataset

In [ ]:
os.sync()
with open(f"{REPO_ROOT}/retrieval/knn_seg_hbird.py", "r") as f:
    t = f.read()
    print(t)

with open(f"{REPO_ROOT}/data/retrieval_dataset.py", "r") as f:
    t = f.read()
    print(t)


%autoreload 2
config_path = "./configs/finetune/finetune_main.yml"
model_config_path = "./configs/finetune/finetune_knn_test.yml"

args = Namespace()

config = utils.merge_config(config_path, args, model_config_path)

config.model_params = utils.merge_model_params_segementation_encoder(
    config, config.encoder_ckpt_model_params)

config.data_path = "./datasets/finetune/epoxy/0.6.57/"

print("Configuration after merge: ", config)

mae = LitMAE.load_from_checkpoint(
    config.encoder_ckpt_path, parameters=config.model_params,
    strict=False
)

config.input_resolution = (1024, 1024)

transform_list = [
    A.PadIfNeeded(
        min_height=config.input_resolution[0],
        min_width=config.input_resolution[1],
        # Avoids reflective padding
        border_mode=cv2.BORDER_CONSTANT,
        value=(0, 0, 0),
        p=1,
    ),
    A.CenterCrop(
        config.input_resolution[0],
        config.input_resolution[1],
    ),
]
transform = A.Compose(
    transform_list,
)

data_module = SemanticDataModule(
    data_path=config.data_path,
    batch_size=16,
    num_workers=8,
    input_resolution=config.input_resolution,
    prob_channel_dropout=0.0,
    prob_channel_swap=0.0,
    classes=config.classes,
    transform=transform,
)

data_module.setup("train")
train_loader = data_module.train_dataloader()
val_loader = data_module.val_dataloader()

encoder = mae.get_encoder()
config.normalize = False
print("Input resolution:", config.input_resolution)
knn_evaluator = KNNHummingBirdSegmentation(encoder=encoder, num_batches=50, 
                                           profile_time=True, normalize=False, 
                                           input_resolution=config.input_resolution,
                                           train_loader=train_loader,
                                           val_loader=val_loader)
torch.cuda.memory_allocated() / (1024**2)

In [ ]:
knn_evaluator.evaluate(betas = [0.02, 0.002, 0.02, 0.02],
                       thresholds = [0.05, 0.05, 0.2, 0.2],
                       k=30,
                       )

In [ ]:
ks = [1, 5, 10, 20, 30, 40, 50]
thresholds = [0.1, 0.05, 0.1, 0.1]
betas = [0.02, 0.002, 0.02, 0.02]


all_results = []
num_classes = 4  # Assuming 4 classes as per the original code

for k in ks:
    for threshold in thresholds:
        for beta in betas:
            print(f"\n=== Testing k={k}, threshold={threshold}, beta={beta} ===")
            
            m_ious = knn_evaluator.evaluate(
                thresholds=[threshold]*4,
                betas=[beta]*4,
                k=k
            )
            
            # Calculate statistics
            for cls in range(num_classes):
                class_scores = m_ious[cls]
                all_results.append({
                    'class': cls,
                    'threshold': threshold,
                    'beta': beta,
                    'fold_scores': class_scores,
                    'mean_miou': np.mean(class_scores),
                    'std_miou': np.std(class_scores),
                    'cv_score': np.mean(class_scores) - np.std(class_scores)  # Conservative estimate
                })
